# 📚 Content-Based Book Recommendation System
### TF-IDF + K-Nearest Neighbors (KNN)

**Dataset:** Best Books Ever (52,478 books)

**Project Goal:** Recommend the Top 5 books that are most similar to a selected book using its **title, author, genres, and description**.

**Approach:**
`Book Dataset → Data Cleaning → Content Features → TF-IDF → KNN → Cosine Distance → Top 5 Recommendations`

This is a **basic AIML mini-project**. No deep learning or complex recommendation algorithms are used.

## 1. Import Libraries
We use:
- Pandas and NumPy for data handling
- Matplotlib and Seaborn for basic EDA
- Scikit-learn for TF-IDF and KNN
- `ast` to safely convert the dataset's list-like text fields

In [ ]:
import pandas as pd
import numpy as np
import ast
import re
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors

pd.set_option("display.max_colwidth", 200)

## 2. Load Dataset

Keep `books_1.Best_Books_Ever.csv` in the same folder as this notebook.

In [ ]:
file_path = "books_1.Best_Books_Ever.csv"

books = pd.read_csv(file_path, low_memory=False)

print("Dataset shape:", books.shape)
books.head()

## 3. Understand the Dataset

In [ ]:
print("Columns:")
print(books.columns.tolist())

print("\nDataset information:")
books.info()

## 4. Select Useful Columns

For a basic **content-based** recommender, we mainly need:
- `title`
- `author`
- `genres`
- `description`

Other columns such as rating, price, cover image, awards, etc. are not required for the core recommendation model.

In [ ]:
books = books[[
    "title",
    "author",
    "genres",
    "description",
    "rating",
    "publisher",
    "language"
]].copy()

books.head()

## 5. Check Missing Values and Duplicates

In [ ]:
print("Missing values:")
print(books[["title", "author", "genres", "description"]].isnull().sum())

print("\nDuplicate rows:", books.duplicated().sum())
print("Duplicate titles:", books["title"].duplicated().sum())

## 6. Data Cleaning

Missing text values are replaced with empty strings. Duplicate book titles are removed so that the recommendation index contains one record per title.

In [ ]:
text_columns = ["title", "author", "genres", "description", "publisher", "language"]

for col in text_columns:
    books[col] = books[col].fillna("").astype(str)

books["title"] = books["title"].str.strip()

# Remove rows without a title
books = books[books["title"] != ""].copy()

# Keep one record per book title
books = books.drop_duplicates(subset="title").reset_index(drop=True)

print("Shape after cleaning:", books.shape)

## 7. Convert Genres into Text

The `genres` column contains list-like values such as:
`['Fantasy', 'Young Adult', 'Adventure']`

We convert them into normal text so TF-IDF can process them.

In [ ]:
def parse_genres(value):
    try:
        parsed = ast.literal_eval(value)
        if isinstance(parsed, list):
            return " ".join(str(x) for x in parsed)
        return str(value)
    except:
        return str(value)

books["genres_text"] = books["genres"].apply(parse_genres)

books[["title", "genres_text"]].head()

## 8. Basic EDA

### Most Common Authors

In [ ]:
top_authors = books["author"].value_counts().head(10)

plt.figure(figsize=(10, 5))
sns.barplot(x=top_authors.values, y=top_authors.index)
plt.title("Top 10 Authors in the Dataset")
plt.xlabel("Number of Books")
plt.ylabel("Author")
plt.show()

### Rating Distribution

In [ ]:
plt.figure(figsize=(8, 5))
sns.histplot(books["rating"].dropna(), bins=20, kde=True)
plt.title("Book Rating Distribution")
plt.xlabel("Rating")
plt.ylabel("Number of Books")
plt.show()

## 9. Create Combined Content

For content-based recommendation, we combine the information that describes a book:

**Title + Author + Genres + Description**

This combined text becomes the input to TF-IDF.

In [ ]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"[^a-zA-Z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

books["combined_text"] = (
    books["title"].apply(clean_text) + " " +
    books["author"].apply(clean_text) + " " +
    books["genres_text"].apply(clean_text) + " " +
    books["description"].apply(clean_text)
)

books[["title", "combined_text"]].head()

## 10. TF-IDF Vectorization

TF-IDF converts the combined book content into numerical vectors.

- Important words receive higher importance.
- Common words receive lower importance.
- Each book becomes a numerical vector that can be compared with other books.

In [ ]:
tfidf = TfidfVectorizer(
    stop_words="english",
    max_features=30000,
    ngram_range=(1, 2)
)

tfidf_matrix = tfidf.fit_transform(books["combined_text"])

print("TF-IDF matrix shape:", tfidf_matrix.shape)

## 11. Build the KNN Model

We use **cosine distance** because it is suitable for comparing TF-IDF text vectors.

`n_neighbors=6` means we retrieve 6 nearest books; one of them is normally the input book itself, so we return the next 5.

In [ ]:
knn = NearestNeighbors(
    n_neighbors=6,
    metric="cosine",
    algorithm="brute"
)

knn.fit(tfidf_matrix)

print("KNN model trained successfully.")

## 12. Create Book Recommendation Function

In [ ]:
# Normalize titles for easier searching
books["title_normalized"] = (
    books["title"]
    .str.lower()
    .str.replace(r"[^a-z0-9\s]", " ", regex=True)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

def recommend_books(book_title, n_recommendations=5):
    query = re.sub(r"[^a-zA-Z0-9\s]", " ", str(book_title).lower())
    query = re.sub(r"\s+", " ", query).strip()

    # Exact match first
    matches = books[books["title_normalized"] == query]

    # If exact match is not found, try partial title matching
    if matches.empty:
        matches = books[books["title_normalized"].str.contains(query, na=False)]

    if matches.empty:
        print("Book not found. Try another title.")
        return None

    book_index = matches.index[0]

    distances, indices = knn.kneighbors(
        tfidf_matrix[book_index],
        n_neighbors=n_recommendations + 1
    )

    recommendations = []

    for distance, index in zip(distances[0], indices[0]):
        if index == book_index:
            continue

        recommendations.append({
            "Title": books.iloc[index]["title"],
            "Author": books.iloc[index]["author"],
            "Genres": books.iloc[index]["genres_text"],
            "Similarity": round(1 - distance, 4)
        })

        if len(recommendations) == n_recommendations:
            break

    return pd.DataFrame(recommendations)

## 13. Test the Recommendation System

In [ ]:
recommend_books("The Hunger Games", 5)

In [ ]:
recommend_books("Harry Potter and the Order of the Phoenix", 5)

In [ ]:
recommend_books("The Hobbit", 5)

## 14. Interactive Recommendation

Enter any book title from the dataset and get the Top 5 similar books.

In [ ]:
book_name = input("Enter a book title: ")

result = recommend_books(book_name, 5)

if result is not None:
    display(result)

## 15. Basic Model Optimization

We test different numbers of neighbors.

Here, `K` means the number of nearest books considered internally. Because the input book itself is included among the nearest neighbors, we request `K + 1` neighbors and then remove the input book.

In [ ]:
k_values = [3, 5, 7, 10]
k_results = []

test_book = "The Hunger Games"

matches = books[books["title_normalized"] == test_book.lower()]

if not matches.empty:
    test_index = matches.index[0]

    for k in k_values:
        model = NearestNeighbors(
            n_neighbors=k + 1,
            metric="cosine",
            algorithm="brute"
        )
        model.fit(tfidf_matrix)

        distances, indices = model.kneighbors(
            tfidf_matrix[test_index],
            n_neighbors=k + 1
        )

        valid_distances = [
            d for d, i in zip(distances[0], indices[0])
            if i != test_index
        ]

        avg_similarity = np.mean([1 - d for d in valid_distances])

        k_results.append({
            "K": k,
            "Average Similarity": round(avg_similarity, 4)
        })

k_results_df = pd.DataFrame(k_results)
k_results_df

### K Comparison

In [ ]:
plt.figure(figsize=(8, 5))
sns.lineplot(
    data=k_results_df,
    x="K",
    y="Average Similarity",
    marker="o"
)
plt.title("K Value vs Average Similarity")
plt.xlabel("K")
plt.ylabel("Average Cosine Similarity")
plt.show()

## 16. Simple Evaluation

A recommendation system does not have a normal classification accuracy score because there is no single correct label for every book.

For this basic project, we use **average cosine similarity** and **genre overlap** as simple evaluation indicators.

- Higher cosine similarity means the recommended books are more similar according to the model.
- Genre overlap checks whether recommended books share at least one genre with the input book.

These are **proxy evaluation measures**, not a formal ground-truth accuracy score.

In [ ]:
def get_genre_set(value):
    try:
        parsed = ast.literal_eval(value)
        if isinstance(parsed, list):
            return {str(x).strip().lower() for x in parsed}
    except:
        pass
    return set()

books["genre_set"] = books["genres"].apply(get_genre_set)

def evaluate_recommendations(book_title, n_recommendations=5):
    query = re.sub(r"[^a-zA-Z0-9\s]", " ", str(book_title).lower())
    query = re.sub(r"\s+", " ", query).strip()

    matches = books[books["title_normalized"] == query]

    if matches.empty:
        return None

    book_index = matches.index[0]
    input_genres = books.iloc[book_index]["genre_set"]

    distances, indices = knn.kneighbors(
        tfidf_matrix[book_index],
        n_neighbors=n_recommendations + 1
    )

    similarities = []
    genre_matches = []

    for distance, index in zip(distances[0], indices[0]):
        if index == book_index:
            continue

        similarities.append(1 - distance)

        recommended_genres = books.iloc[index]["genre_set"]
        genre_matches.append(
            len(input_genres.intersection(recommended_genres)) > 0
        )

        if len(similarities) == n_recommendations:
            break

    return {
        "Book": books.iloc[book_index]["title"],
        "Average Cosine Similarity": round(np.mean(similarities), 4),
        "Genre Overlap Rate": round(np.mean(genre_matches), 4)
    }

evaluate_recommendations("The Hunger Games", 5)

## 17. Final Recommendation Demo

In [ ]:
book_name = "The Hunger Games"

print("Input Book:", book_name)
print("\nTop 5 Recommended Books:\n")

display(recommend_books(book_name, 5))

print("\nEvaluation:")
print(evaluate_recommendations(book_name, 5))

## 18. Result Analysis

### Expected Result
The system should recommend books that have similar:
- genres
- authors
- important words in the description
- title-related terms

### Key Observation
Because this is a **content-based** system, recommendations are based on the content/features of books rather than user-user behavior.

### Limitation
The system does not understand the meaning of a story like a human. It uses TF-IDF word importance and cosine distance, so recommendations depend on the text available in the dataset.

## 19. Conclusion

A basic **Content-Based Book Recommendation System** was developed using **TF-IDF and K-Nearest Neighbors**.

The model converts book information into numerical TF-IDF vectors and uses cosine distance to identify the most similar books. The system can take a book title as input and return the Top 5 similar books.

This approach is simple, explainable, and suitable for an AIML mini-project and viva.